# Visual Contrastive Decoding (VCD) - CHAIR Benchmark (Google Colab & Kaggle)

This notebook runs the **CHAIR (Caption Hallucination Assessment with Image Relevance)** benchmark on **10 images sampled from COCO val2014 (seed: 2027)** using **Visual Contrastive Decoding (VCD)** to profile accuracy and **calculate average runtime per sample**.

### Benchmark Settings:
- **Evaluation Dataset**: 10 images sampled from COCO val2014 with **Seed: 2027** (`selected_chair_val2014_seed2027.json`)
- **Prompt**: `"Describe this image."`
- **Generation**: `max_new_tokens = 128`, Greedy Decoding (`do_sample = False`, `temperature = 0.0`)
- **Evaluation Code**: Standalone CHAIR metric ([Maxlinn/CHAIR-metric-standalone](https://github.com/Maxlinn/CHAIR-metric-standalone/tree/main))
- **Reported Metrics**:
  1. `CHAIRs` (%): Sentence-level hallucination rate
  2. `CHAIRi` (%): Instance-level hallucination rate
  3. `Recall` (%): Ground-truth object recall
  4. `Caption Length`: Average caption length (words)
  5. `Avg Time / Sample` (s): Measured accurately with `import time` and GPU synchronization
  6. `Total Time` (s): Total inference execution time
- **Supported Models**: `llava` (`llava-hf/llava-1.5-7b-hf`) and `qwen2vl` (`Qwen/Qwen2-VL-7B-Instruct`)
- **Methods**: Visual Contrastive Decoding (`--use_vcd`) vs Standard Baseline (`--no_vcd`)

---
### Setup Instructions for Google Colab:
1. **GPU**: Select **Runtime -> Change runtime type -> T4 GPU** (or A100/L4).
2. **Secrets (🔑 icon on the left panel)**:
   - `HF_TOKEN`: Hugging Face access token.
   - `KAGGLE_USERNAME`: Your Kaggle username.
   - `KAGGLE_KEY`: Your Kaggle API key (from `kaggle.json`).
3. **Data**: Cell 4 will automatically download and extract `biminhco/val2014` from Kaggle into `/content/data`.

### Setup Instructions for Kaggle:
1. **Accelerator**: Select **GPU T4 x2**, toggle **Internet ON**.
2. **Secrets**: Add `HF_TOKEN` in Add-ons -> Secrets.
3. **Dataset**: Mount `datasets/biminhco/val2014/val2014` in **+ Add Input**.


In [ ]:
# Cell 1: Environment Setup & Dependencies Installation
import os
os.environ['PYTHONUNBUFFERED'] = '1'

# 1. Gỡ bỏ torchaudio để tránh xung đột CUDA
!pip uninstall -y -q torchaudio

# 2. Cài đặt các thư viện cần thiết (bao gồm cả kaggle CLI)
!pip install -q --no-cache-dir \
    "transformers>=4.45.0" \
    "accelerate>=0.26.0" \
    sentencepiece \
    protobuf \
    tiktoken \
    qwen_vl_utils \
    pyyaml \
    tqdm \
    nltk \
    huggingface_hub \
    pandas \
    kaggle

print("✅ Dependencies successfully installed!")
from transformers import AutoProcessor
print("✅ AutoProcessor import verified successfully!")


In [ ]:
# Cell 2: Authenticate Hugging Face & Kaggle Secrets
import os
import sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
IN_KAGGLE = os.path.exists('/kaggle')

# 1. Hugging Face Login
try:
    hf_token = None
    if IN_COLAB:
        from google.colab import userdata
        try:
            hf_token = userdata.get('HF_TOKEN')
        except Exception:
            pass
    elif IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        try:
            hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    if not hf_token:
        hf_token = os.environ.get("HF_TOKEN")

    if hf_token:
        import huggingface_hub
        huggingface_hub.login(token=hf_token)
        print("✅ Successfully logged in to Hugging Face Hub!")
    else:
        print("[Notice] HF_TOKEN not found in Secrets. (Ensure it's added if accessing gated models)")
except Exception as e:
    print(f"[Notice] Hugging Face login info: {e}")

# 2. Kaggle API Configuration (for downloading data when running on Colab)
if IN_COLAB:
    if os.path.exists("/content/kaggle.json"):
        !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
        print("✅ Configured Kaggle API using /content/kaggle.json")
    else:
        try:
            from google.colab import userdata
            k_user = userdata.get('KAGGLE_USERNAME')
            k_key = userdata.get('KAGGLE_KEY')
            if k_user and k_key:
                os.environ['KAGGLE_USERNAME'] = k_user
                os.environ['KAGGLE_KEY'] = k_key
                print("✅ Kaggle API credentials configured from Colab Secrets!")
            else:
                print("[Notice] KAGGLE_USERNAME / KAGGLE_KEY not found in Colab Secrets.")
        except Exception as e:
            print(f"[Notice] Kaggle credentials check: {e}")


In [ ]:
# Cell 3: Clone Repository or Pull Latest Code
import os
import sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
repo_path = "/content/VCD" if IN_COLAB else "/kaggle/working/VCD"

if not os.path.exists(repo_path):
    !git clone https://github.com/ntmy12/VCD.git {repo_path}
else:
    print(f"Pulling latest changes in {repo_path}...")
    !cd {repo_path} && git pull origin master

# Enter vcd_experiments directory
%cd {repo_path}/vcd_experiments


In [ ]:
# Cell 4: Download COCO val2014 from Kaggle (Active on Colab)
import os
import sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    data_dir = "/content/data"
    os.makedirs(data_dir, exist_ok=True)
    sample_img = "COCO_val2014_000000102421.jpg"
    
    # Check if images already exist (e.g. from previous run)
    found_img = False
    for r, _, files in os.walk(data_dir):
        if sample_img in files:
            found_img = True
            print(f"✅ COCO val2014 images already present in: {r}")
            break
            
    if not found_img:
        print("📥 Downloading COCO val2014 dataset from Kaggle (biminhco/val2014)...")
        res = !kaggle datasets download -d biminhco/val2014 -p {data_dir} --unzip
        
        # Check if Kaggle download was successful
        for r, _, files in os.walk(data_dir):
            if sample_img in files:
                found_img = True
                print(f"✅ Downloaded & extracted successfully to: {r}")
                break
                
        # Fallback to direct official COCO mirror if Kaggle download was not available
        if not found_img:
            print("⚠️ Kaggle download not available or failed. Falling back to direct COCO val2014 mirror...")
            !wget -c http://images.cocodataset.org/zips/val2014.zip -P {data_dir}
            !unzip -q -n {data_dir}/val2014.zip -d {data_dir}
            print("✅ COCO val2014 downloaded and extracted via mirror!")
else:
    print("Running on Kaggle: Dataset is attached via /kaggle/input.")


In [ ]:
# Cell 5: Hardware & Environment Verification
import torch
import yaml
import os
import sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

print("=== GPU Environment ===")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print(f"  Device {i}: {prop.name} ({prop.total_memory / 1e9:.2f} GB VRAM)")

print("\n=== CHAIR Verification ===")
config_file = "configs/data_paths_colab.yaml" if IN_COLAB else "configs/data_paths_kaggle.yaml"
if not os.path.exists(config_file):
    config_file = "configs/data_paths.yaml"

with open(config_file, "r") as f:
    cfg = yaml.safe_load(f)

coco_img_dir = cfg.get("coco_val2014_images", "")
manifest_path = cfg.get("chair_manifest", "benchmarks/chair/selected_chair_val2014_seed2027.json")
cache_path = cfg.get("chair_eval_cache", "benchmarks/chair/chair.pkl")

print(f"Selected Config:      {config_file}")
print(f"Configured Image Dir: {coco_img_dir}")
print(f"Manifest Path:        {manifest_path} (Exists: {os.path.isfile(manifest_path)})")
print(f"Evaluator Cache:      {cache_path} (Exists: {os.path.isfile(cache_path)})")


In [ ]:
# Cell 6: Run CHAIR Benchmark Evaluation (10 Samples for Latency Profiling)
import os
import sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
config_file = 'configs/data_paths_colab.yaml' if IN_COLAB else 'configs/data_paths_kaggle.yaml'

# Select model: 'llava' (LLaVA-1.5-7B) or 'qwen2vl' (Qwen2-VL-7B-Instruct)
MODEL = 'llava'

# Quick benchmark: 10 samples to calculate average runtime per sample
NUM_SAMPLES = 10

# Resume & Real-Time Logging Controls:
# - RESUME = True: Auto-detects and resumes from the latest run in results/ if previously stopped
# - RESUME_FROM = '': Or specify exact path to raw_outputs.jsonl or result folder
# - LOG_INTERVAL = 2: Emits newline ('\n') log every N images with flush=True to bypass buffer
RESUME = True
RESUME_FROM = ''
LOG_INTERVAL = 2

resume_flag = '--resume' if (RESUME and not RESUME_FROM) else ''
resume_from_flag = f'--resume_from {RESUME_FROM}' if RESUME_FROM else ''

# Notice the '-u' flag passed to python: forces unbuffered stdout/stderr on Kaggle/Colab
!python -u benchmarks/chair/run_chair.py \
    --model {MODEL} \
    --use_vcd \
    --config_path {config_file} \
    --seed 2027 \
    --num_samples {NUM_SAMPLES} \
    --max_new_tokens 128 \
    --prompt 'Describe this image.' \
    --noise_step 500 \
    --cd_alpha 1.0 \
    --cd_beta 0.1 \
    --dtype fp16 \
    --log_interval {LOG_INTERVAL} \
    {resume_flag} \
    {resume_from_flag}


In [ ]:
# Cell 7: Standalone CHAIR Evaluation from raw_outputs.jsonl (Zero GPU / Instant)
# Run this cell to evaluate or recompute CHAIR metrics from raw_outputs.jsonl directly without loading the model:
!python -u benchmarks/chair/eval_chair.py


In [ ]:
# Cell 8: Summarize, Calculate Average Runtime, and Display Results
import os
import glob
import json
import time
import pandas as pd

# Search all chair result directories
run_dirs = sorted(glob.glob("results/*chair*"))
if run_dirs:
    latest_dir = run_dirs[-1]
    raw_file = os.path.join(latest_dir, "raw_outputs.jsonl")
    metrics_file = os.path.join(latest_dir, "metrics.json")
    config_file = os.path.join(latest_dir, "run_config.json")
    
    cfg = {}
    if os.path.exists(config_file):
        with open(config_file, "r") as f:
            cfg = json.load(f)
        print(f"Run Directory: {latest_dir}")
        print(f"Model: {cfg.get('model')} | VCD: {cfg.get('use_vcd')} | Max Tokens: {cfg.get('max_new_tokens')} | Seed: {cfg.get('seed')}")

    # Auto-evaluate if raw_outputs exists but metrics.json is missing
    if not os.path.exists(metrics_file) and os.path.exists(raw_file):
        print("[Notice] Computing CHAIR metrics for raw_outputs.jsonl...")
        !python -u benchmarks/chair/eval_chair.py --results_file {raw_file}

    if os.path.exists(metrics_file):
        with open(metrics_file, "r") as f:
            metrics = json.load(f)
        
        # Fallback latency calculation if not recorded in metrics
        if "avg_time_per_sample_s" not in metrics and os.path.exists(raw_file):
            latencies = []
            with open(raw_file, "r", encoding="utf-8") as rf:
                for line in rf:
                    rec = json.loads(line.strip())
                    if "latency_s" in rec:
                        latencies.append(rec["latency_s"])
            if latencies:
                metrics["total_inference_time_s"] = round(sum(latencies), 4)
                metrics["avg_time_per_sample_s"] = round(sum(latencies) / len(latencies), 4)

        num_eval = metrics.get('num_evaluated', metrics.get('total_evaluated', 10))
        tot_time = metrics.get('total_inference_time_s', 0.0)
        avg_time = metrics.get('avg_time_per_sample_s', 0.0)

        df = pd.DataFrame([{
            "Model": str(cfg.get('model', MODEL)).upper(),
            "Method": 'VCD' if cfg.get('use_vcd', True) else 'Baseline',
            "Samples": num_eval,
            "CHAIRs (%)": round(metrics.get('CHAIRs', 0.0), 2),
            "CHAIRi (%)": round(metrics.get('CHAIRi', 0.0), 2),
            "Recall (%)": round(metrics.get('Recall', 0.0), 2),
            "Caption Length (words)": round(metrics.get('Caption_Length', 0.0), 2),
            "Avg Time/Sample (s)": avg_time,
            "Total Time (s)": tot_time
        }])
        print(f"\n{'='*32} CHAIR BENCHMARK METRICS (10 SAMPLES) {'='*32}")
        display(df)
        print('='*95)

        # Detailed Timing Summary Card
        print("\n⏱️ " + "="*68)
        print(f"   AVERAGE RUNTIME PER SAMPLE SUMMARY | {str(cfg.get('model', MODEL)).upper()} + {'VCD' if cfg.get('use_vcd', True) else 'Baseline'} (CHAIR)")
        print("="*71)
        print(f"   • Total Images Evaluated  : {num_eval} samples (Seed 2027)")
        print(f"   • Total Inference Time    : {tot_time:.2f} seconds")
        print(f"   • AVERAGE TIME / SAMPLE   : {avg_time:.4f} seconds / sample")
        if avg_time > 0:
            print(f"   • Throughput              : {1.0 / avg_time:.2f} samples / second")
        print("="*71)
    
    if os.path.exists(raw_file):
        print(f"\n{'='*30} SAMPLE GENERATED CAPTIONS {'='*30}")
        with open(raw_file, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5: break
                item = json.loads(line)
                lat_str = f" | Latency: {item['latency_s']}s" if 'latency_s' in item else ""
                print(f"[{i+1}] Image {item['file_name']} (ID: {item['image_id']}{lat_str}):")
                print(f"    Caption: {item['caption']}\n")

    # Cross-run comparison table
    all_summary_files = sorted(glob.glob("results/*chair*/summary_metrics.json"))
    if len(all_summary_files) > 1:
        comp_rows = []
        for s_file in all_summary_files:
            folder = os.path.basename(os.path.dirname(s_file))
            try:
                with open(s_file, 'r', encoding='utf-8') as f:
                    s_data = json.load(f)
                m_info = s_data.get("metrics", {})
                comp_rows.append({
                    "Model": str(s_data.get("model", "")).upper(),
                    "Method": 'VCD' if s_data.get("use_vcd", True) else 'Baseline',
                    "Samples": m_info.get("num_evaluated", m_info.get("total_evaluated", 0)),
                    "CHAIRs (%)": m_info.get("CHAIRs", 0.0),
                    "CHAIRi (%)": m_info.get("CHAIRi", 0.0),
                    "Recall (%)": m_info.get("Recall", 0.0),
                    "Avg Time/Sample (s)": m_info.get("avg_time_per_sample_s", 0.0),
                    "Total Time (s)": m_info.get("total_inference_time_s", 0.0),
                    "Run Dir": folder
                })
            except Exception:
                pass
        if comp_rows:
            print("\n📊 Cross-Run Comparison (CHAIR):")
            display(pd.DataFrame(comp_rows))
else:
    print("No results found in results/ directory yet.")
